In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from statsmodels.stats.proportion import proportion_confint
import xgboost as xgb
import torch

In [ ]:
# Photosynthetic pathway (C3/C4) per reference species.
# Index aligns with the integer species labels in Poaceae_reference_labels.
#
# Species names were transcribed from the original microscope slide labels of the
# reference pollen collections and subsequently rectified against Plants of the
# World Online (POWO; https://powo.science.kew.org/) to correct transcription errors and misspellings and
# to update synonymies to currently accepted names. Inline notes document these
# modifications as well as pathway reassignments based on prior literature.


photosynthetic_pathway = [
  "C3",  # Agropyron ciliare (= Elymus ciliaris)
  "C4",  # Agrostis mexicana (= Agrostis tolucensis); reassigned to Eragrostis (C4)
  "C3",  # Agrostis quinqueseta
  "C3",  # Agrostis trachyphylla
  "C3",  # Agrostis volkensii
  "C3",  # Amphibromus neesii
  "C4",  # Andropogon amethystinus
  "C4",  # Andropogon chrysostachyus
  "C4",  # Andropogon contortus (= Heteropogon contortus)
  "C4",  # Andropogon lima
  "C4",  # Andropogon schirensis; reassigned to C4 (potential error in Wooller et al. 2001)
  "C3",  # Anthoxanthum nivale
  "C4",  # Aristida implexa (= Aristida megapotamica)
  "C4",  # Bothriochloa bladhii
  "C4",  # Brachiaria brizantha (= Urochloa brizantha)
  "C3",  # Brachypodium flexum
  "C3",  # Bromus auleticus
  "C3",  # Bromus brachyphyllus (= Bromus orcuttianus)
  "C3",  # Bromus ciliatus
  "C3",  # Bromus lanatus
  "C3",  # Bromus leptoclados
  "C3",  # Bromus thominei (= Bromus hordeaceus subsp. thominei)
  "C3",  # Calamagrostis epigejos
  "C4",  # Chrysopogon fallax
  "C4",  # Cymbopogon nardus
  "C4",  # Cynodon dactylon
  "C3",  # Dactylis glomerata
  "C4",  # Digitaria abyssinica
  "C3",  # Echinopogon caespitosus
  "C3",  # Ehrharta erecta
  "C3",  # Ehrharta erecta var. natalensis; retained as C3 (C4 assignment unsupported)
  "C4",  # Eleusine coracana
  "C4",  # Eleusine jaegeri
  "C4",  # Exotheca abyssinica
  "C3",  # Festuca africana (= Pseudobromus africanus)
  "C3",  # Festuca costata
  "C3",  # Festuca elatior (= Ampelodesmos mauritanicus)
  "C3",  # Isachne mauritiana
  "C3",  # Koeleria capensis
  "C3",  # Lolium perenne
  "C3",  # Melica onoei
  "C4",  # Miscanthus violaceus (= Miscanthidium violaceum)
  "C3",  # Oplismenus compositus
  "C3",  # Oplismenus hirtellus
  "C4",  # Pennisetum longistylum (= Cenchrus clandestinus)
  "C3",  # Pentaschistis borussica (= Pentameris borussica)
  "C3",  # Phalaris arundinacea
  "C3",  # Poa anceps
  "C3",  # Poa leptoclada
  "C3",  # Poa schimperiana
  "C4",  # Saccharum arundinaceum (= Tripidium arundinaceum)
  "C3",  # Secale cereale
  "C4",  # Setaria megaphylla
  "C3",  # Sinarundinaria alpina (= Oldeania alpina)
  "C4",  # Sorghum halepense
  "C4",  # Spartina pectinata (= Sporobolus michauxianus)
  "C3",  # Stipa compacta (= Austrostipa flavescens)
  "C3",  # Streblochaete longiarista (= Koordersiochloa longiarista)
  "C4",  # Themeda triandra
  "C4"   # Zea mays
]

In [ ]:
import pandas as pd

# Load modern CNN features and species labels
features_path = "E:Grass_Project/Concatenated_modern_rutundu_FixMatch.csv"
labels_path = "E:Grass_Project/Image_labels_modern_Rutundu_FixMatch.csv"

df_modern_Poaceae_features = pd.read_csv(features_path, header=None)
df_modern_Poaceae_labels = pd.read_csv(labels_path, header=None)

Poaceae_reference_labels = df_modern_Poaceae_labels[0].values

In [ ]:
# Assign photosynthetic pathway per specimen
specimen_pathway = np.array(photosynthetic_pathway)[Poaceae_reference_labels]
y = np.where(specimen_pathway == "C4", 1, 0)
X = df_modern_Poaceae_features.values

In [ ]:
# Standardize features and reduce dimensionality with PCA (95% variance)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

pca = PCA(n_components=0.95, random_state=123)
X_pca = pca.fit_transform(X_scaled)

In [ ]:
# Train/validation split to evaluate classifier performance
X_train, X_val, y_train, y_val = train_test_split(
    X_pca, y, test_size=0.2, stratify=y, random_state=42
)

xgb_params = dict(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=5,
    min_child_weight=2,
    subsample=0.8,
    colsample_bytree=0.9,
    reg_alpha=0.1,
    reg_lambda=1.0,
    eval_metric="logloss",
    n_jobs=-1,
    random_state=42
)

model_val = xgb.XGBClassifier(**xgb_params)
model_val.fit(X_train, y_train)

y_pred_val = model_val.predict(X_val)
val_accuracy = accuracy_score(y_val, y_pred_val)

print(f"Validation accuracy: {val_accuracy:.3f}")
print(classification_report(y_val, y_pred_val, target_names=["C3", "C4"]))
print(confusion_matrix(y_val, y_pred_val))

In [ ]:
# Retrain on all modern specimens before fossil inference
model = xgb.XGBClassifier(**xgb_params)
model.fit(X_pca, y)

In [ ]:
# Map fossil core depths to 14C ages (rounded to nearest 5 years)

Y = [20,20,20,20,20,20,20,20,20,20,20,20,20,20,20,20,20,20,20,20,20,20,20,20,20,20,20,20,20,20,20,20,20,20,20,20,20,20,20,20,20,20,20,20,20,20,20,20,20,39,39,39,39,39,39,39,39,39,39,39,39,39,39,39,39,39,39,39,39,39,39,39,39,39,39,39,39,39,39,39,39,39,39,39,39,39,39,39,39,39,39,39,39,39,39,39,39,39,39,39,71,71,71,71,71,71,71,71,71,71,71,71,71,71,71,71,71,71,71,71,71,71,71,71,71,71,71,71,71,71,71,71,71,71,71,71,71,71,71,71,71,71,71,71,71,71,71,71,71,71,71,71,71,71,124,124,124,124,124,124,124,124,124,124,124,124,124,124,124,124,124,124,124,124,124,124,124,124,124,124,124,124,124,124,124,124,124,124,124,124,124,124,124,124,124,124,124,124,124,124,124,124,124,124,124,134,134,134,134,134,134,134,134,134,134,134,134,134,134,134,134,134,134,134,134,134,134,134,134,134,134,134,134,134,134,134,134,134,134,134,134,134,134,134,134,134,134,134,134,134,134,134,134,134,134,134,163,163,163,163,163,163,163,163,163,163,163,163,163,163,163,163,163,163,163,163,163,163,163,163,163,163,163,163,163,163,163,163,163,163,163,163,163,163,163,163,163,163,163,163,163,163,163,163,163,163,163,173,173,173,173,173,173,173,173,173,173,173,173,173,173,173,173,173,173,173,173,173,173,173,173,173,173,173,173,173,173,173,173,173,173,173,173,173,173,173,173,173,173,173,173,173,173,173,173,173,173,195,195,195,195,195,195,195,195,195,195,195,195,195,195,195,195,195,195,195,195,195,195,195,195,195,195,195,195,195,195,195,195,195,195,195,195,195,195,195,195,195,195,195,195,195,195,195,195,195,195,195,236,236,236,236,236,236,236,236,236,236,236,236,236,236,236,236,236,236,236,236,236,236,236,236,236,236,236,236,236,236,236,236,236,236,236,236,236,236,236,236,236,236,236,236,236,236,236,236,236,256,256,256,256,256,256,256,256,256,256,256,256,256,256,256,256,256,256,256,256,256,256,256,256,256,256,256,256,256,256,256,256,256,256,256,256,256,256,256,256,256,256,256,256,256,256,256,256,256,256,263,263,263,263,263,263,263,263,263,263,263,263,263,263,263,263,263,263,263,263,263,263,263,263,263,263,263,263,263,263,263,263,263,263,263,263,263,263,263,263,263,263,263,263,263,263,263,263,263,286,286,286,286,286,286,286,286,286,286,286,286,286,286,286,286,286,286,286,286,286,286,286,286,286,286,286,286,286,286,286,286,286,286,286,286,286,286,286,286,286,286,286,286,286,286,286,286,286,286,298,298,298,298,298,298,298,298,298,298,298,298,298,298,298,298,298,298,298,298,298,298,298,298,298,298,298,298,298,298,298,298,298,298,298,298,298,298,298,298,298,298,298,298,298,298,298,298,298,298,531,531,531,531,531,531,531,531,531,531,531,531,531,531,531,531,531,531,531,531,531,531,531,531,531,531,531,531,531,531,531,531,531,531,531,531,531,531,531,531,531,531,531,531,531,531,531,531,531,531,321,321,321,321,321,321,321,321,321,321,321,321,321,321,321,321,321,321,321,321,321,321,321,321,321,321,321,321,321,321,321,321,321,321,321,321,321,321,321,321,321,321,321,321,321,321,321,321,321,321,335,335,335,335,335,335,335,335,335,335,335,335,335,335,335,335,335,335,335,335,335,335,335,335,335,335,335,335,335,335,335,335,335,335,335,335,335,335,335,335,335,335,335,335,335,335,335,335,335,335,375,375,375,375,375,375,375,375,375,375,375,375,375,375,375,375,375,375,375,375,375,375,375,375,375,375,375,375,375,375,375,375,375,375,375,375,375,375,375,375,375,375,375,375,375,375,375,375,375,375,397,397,397,397,397,397,397,397,397,397,397,397,397,397,397,397,397,397,397,397,397,397,397,397,397,397,397,397,397,397,397,397,397,397,397,397,397,397,397,397,397,397,397,397,397,397,397,397,397,423,423,423,423,423,423,423,423,423,423,423,423,423,423,423,423,423,423,423,423,423,423,423,423,423,423,423,423,423,423,423,423,423,423,423,423,423,423,423,423,423,423,423,423,423,423,423,423,423,423,432,432,432,432,432,432,432,432,432,432,432,432,432,432,432,432,432,432,432,432,432,432,432,432,432,432,432,432,432,432,432,432,432,432,432,432,432,432,432,432,432,432,432,432,432,432,432,432,453,453,453,453,453,453,453,453,453,453,453,453,453,453,453,453,453,453,453,453,453,453,453,453,453,453,453,453,453,453,453,453,453,453,453,453,453,453,453,453,453,453,453,453,453,453,453,453,453,470,470,470,470,470,470,470,470,470,470,470,470,470,470,470,470,470,470,470,470,470,470,470,470,470,470,470,470,470,470,470,470,470,470,470,470,470,470,470,470,470,470,470,470,470,470,470,470,470,470,470,483,483,483,483,483,483,483,483,483,483,483,483,483,483,483,483,483,483,483,483,483,483,483,483,483,483,483,483,483,483,483,483,483,483,483,483,483,483,483,483,483,483,483,483,483,483,483,483,483,483,483,516,516,516,516,516,516,516,516,516,516,516,516,516,516,516,516,516,516,516,516,516,516,516,516,516,516,516,516,516,516,516,516,516,516,516,516,516,516,516,516,516,516,516,516,516,516,516,516,516,516]

depth_to_age = {
    20: 252.922, 39: 1006.79, 71: 2446.333, 124: 4497.479, 134: 5001.556,
    163: 6496.673, 173: 7022.491, 195: 8195.662, 236: 9281.874, 256: 10554.34,
    263: 11004.19, 286: 12497.5, 298: 13285.34, 321: 14230.91, 335: 14995.26,
    375: 17204.36, 397: 18434.3, 423: 19520.83, 432: 20040.09, 453: 21257.91,
    470: 22249.94, 483: 23012.15, 516: 24014.17, 531: 25000.81,
}

depth_to_age = {d: 5 * round(a / 5) for d, a in depth_to_age.items()}

Y_tens = torch.tensor([depth_to_age[d] for d in Y])

In [ ]:
# Load fossil features and apply the same preprocessing steps
fossil_features_path = "E:Grass_Project/Rutundu_fossils_MIPs_patches_concatenated_FixMatch.csv"

df_Rutundu_fossils = pd.read_csv(fossil_features_path, header=None)
X_Rutundu_fossils = df_Rutundu_fossils.to_numpy()

X_fossil_scaled = scaler.transform(X_Rutundu_fossils)
X_fossil_pca = pca.transform(X_fossil_scaled)

In [ ]:
# Predict C3/C4 for each fossil specimen
fossil_predictions = model.predict(X_fossil_pca)
fossil_probabilities = model.predict_proba(X_fossil_pca)

In [ ]:
# Aggregate predictions by core depth (14C age) and compute Wilson CIs
fossil_ages = Y_tens.numpy() if hasattr(Y_tens, "numpy") else np.array(Y_tens)

results = pd.DataFrame({
    "age": fossil_ages,
    "prediction": fossil_predictions,
    "c4_probability": fossil_probabilities[:, 1],
})

age_summary = []
for age, group in results.groupby("age"):
    n = len(group)
    n_c4 = int((group["prediction"] == 1).sum())
    c4_prop = n_c4 / n
    ci_low, ci_high = proportion_confint(n_c4, n, method="wilson", alpha=0.05)
    age_summary.append({
        "age": age,
        "n_specimens": n,
        "c4_count": n_c4,
        "c4_proportion": c4_prop,
        "c4_ci_lower": ci_low,
        "c4_ci_upper": ci_high,
    })

age_summary = pd.DataFrame(age_summary).sort_values("age", ascending=False).reset_index(drop=True)
print(age_summary)

In [ ]:
import matplotlib.pyplot as plt
from scipy.stats import linregress

Age_XGB_C4 = np.array([255, 1005, 2445, 4495, 5000, 6495, 7020, 8195, 9280, 10555, 11005, 12500, 13285, 14230, 14995, 17205, 18435, 19520, 20040, 21260, 22250, 23010, 24015, 25000])
Mean_XGB_C4 = np.array([0.265306, 0.411765, 0.370370, 0.392157, 0.686275, 0.607843, 0.540000, 0.431373, 0.530612, 0.480000, 0.469388, 0.500000, 0.640000, 0.480000, 0.640000, 0.620000, 0.469388, 0.280000, 0.583333, 0.489796, 0.490196, 0.627451, 0.440000, 0.400000])*100
Lower_XGB_C4 = np.array([0.162113, 0.287544, 0.254234, 0.270273, 0.549730, 0.470852, 0.403989, 0.305012, 0.393809, 0.347971, 0.337035, 0.366445, 0.501410, 0.347971, 0.501410, 0.481504, 0.337035, 0.174742, 0.442813, 0.355751, 0.358575, 0.490252, 0.311622, 0.276084])*100
Upper_XGB_C4= np.array([0.402623, 0.548347, 0.503725, 0.529148, 0.796723, 0.729727, 0.670303, 0.567347, 0.662965, 0.614883, 0.606191, 0.633555, 0.758613, 0.614883, 0.758613, 0.741372, 0.606191, 0.416651, 0.711503, 0.625324, 0.623191, 0.746795, 0.576940, 0.538186])*100

# Current run values from age_summary (convert proportions to percent)
current_age = age_summary["age"].to_numpy()
current_mean = age_summary["c4_proportion"].to_numpy() * 100
current_lower = age_summary["c4_ci_lower"].to_numpy() * 100
current_upper = age_summary["c4_ci_upper"].to_numpy() * 100

# Align current run to reference ages
order = np.argsort(current_age)
current_age_sorted = current_age[order]
current_mean_sorted = current_mean[order]
current_lower_sorted = current_lower[order]
current_upper_sorted = current_upper[order]

ref_order = np.argsort(Age_XGB_C4)
ref_age_sorted = Age_XGB_C4[ref_order]
ref_mean_sorted = Mean_XGB_C4[ref_order]
ref_lower_sorted = Lower_XGB_C4[ref_order]
ref_upper_sorted = Upper_XGB_C4[ref_order]

# Regression of current vs reference at matched ages
common_ages, ref_idx, cur_idx = np.intersect1d(ref_age_sorted, current_age_sorted,
                                                return_indices=True)
ref_matched = ref_mean_sorted[ref_idx]
cur_matched = current_mean_sorted[cur_idx]
slope, intercept, r_value, p_value, _ = linregress(ref_matched, cur_matched)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Panel 1: C4 proportion through time
ax = axes[0]
ax.errorbar(ref_age_sorted, ref_mean_sorted,
            yerr=[ref_mean_sorted - ref_lower_sorted, ref_upper_sorted - ref_mean_sorted],
            fmt="o-", capsize=3, color="black", label="Published", alpha=0.8)
ax.errorbar(current_age_sorted, current_mean_sorted,
            yerr=[current_mean_sorted - current_lower_sorted,
                  current_upper_sorted - current_mean_sorted],
            fmt="s--", capsize=3, color="firebrick", label="Current run", alpha=0.8)
ax.set_xlabel("14C age (yr BP)")
ax.set_ylabel("C4 (%)")
ax.invert_xaxis()
ax.legend()
ax.grid(alpha=0.3)

# Panel 2: current vs published at matched ages
ax = axes[1]
lims = [0, 100]
ax.plot(lims, lims, "k--", alpha=0.5, label="1:1")
ax.plot(lims, intercept + slope * np.array(lims), "-", color="firebrick",
        label=f"y = {slope:.2f}x + {intercept:.2f}\nr = {r_value:.3f}, p = {p_value:.2e}")
ax.scatter(ref_matched, cur_matched, color="black", zorder=3)
ax.set_xlabel("Published C4 (%)")
ax.set_ylabel("Current run C4 (%)")
ax.set_xlim(lims)
ax.set_ylim(lims)
ax.set_aspect("equal")
ax.legend(loc="upper left")
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

# Quantitative comparison
mae = np.mean(np.abs(cur_matched - ref_matched))
print(f"Matched ages: {len(common_ages)}")
print(f"Mean absolute deviation: {mae:.2f} percentage points")
print(f"Pearson r: {r_value:.3f} (p = {p_value:.2e})")
print(f"Regression slope: {slope:.3f}, intercept: {intercept:.3f}")